# 5주차 — SELECT ③ 계산·CASE·집계
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다. 원본은 언제든 다시 내려받을 수 있으니 마음껏 실험하세요.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 조회 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, urllib.request, sqlite3
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/mini_emr/mini_emr.sqlite"   # 강의자 공지 주소
if not os.path.exists("mini_emr.sqlite"):
    try:
        urllib.request.urlretrieve(DB_URL, "mini_emr.sqlite")
        print("mini_emr.sqlite 다운로드 완료")
    except Exception as e:
        print("자동 다운로드 실패 — 왼쪽 파일 탭에서 mini_emr.sqlite를 직접 업로드하세요.", e)

con = sqlite3.connect("mini_emr.sqlite")

def q(sql):
    """SQL을 실행하고 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """결과를 반환하지 않는 SQL(INSERT/UPDATE/CREATE 등) 실행"""
    con.executescript(sql); con.commit(); print("실행 완료")

q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 1. 산술식 — 재원일수

In [ ]:
# 입원별 재원일수(LOS)
q("""
SELECT visit_id, visit_date, discharge_date,
       julianday(discharge_date) - julianday(visit_date) AS los
FROM visit WHERE visit_type = '입원' ORDER BY los DESC LIMIT 8
""")

## 2. CASE — 혈압 분류

In [ ]:
# 정상/주의/고혈압
q("""
SELECT visit_id, sbp, dbp,
  CASE WHEN sbp >= 140 OR dbp >= 90 THEN '고혈압'
       WHEN sbp >= 120 THEN '주의'
       ELSE '정상' END AS bp_grade
FROM vital ORDER BY sbp DESC LIMIT 10
""")

## 3. 집계함수

In [ ]:
# 입원 요약: 건수·평균·최장
q("""
SELECT COUNT(*) AS 입원건수,
       ROUND(AVG(julianday(discharge_date) - julianday(visit_date)), 1) AS 평균재원일,
       MAX(julianday(discharge_date) - julianday(visit_date)) AS 최장재원일
FROM visit WHERE visit_type = '입원'
""")

In [ ]:
# 정상범위 벗어난 검사 비율 (CASE×SUM 패턴)
q("""
SELECT COUNT(*) AS 전체검사,
  SUM(CASE WHEN result_value > ref_high OR result_value < ref_low THEN 1 ELSE 0 END) AS 이상건수,
  ROUND(100.0 * SUM(CASE WHEN result_value > ref_high OR result_value < ref_low THEN 1 ELSE 0 END) / COUNT(*), 1) AS 이상비율
FROM lab
""")

In [ ]:
# COUNT(*) vs COUNT(컬럼) — NULL의 조용한 제외
q("""
SELECT COUNT(*) AS 전체행수, COUNT(result_value) AS 값있는행수,
       COUNT(*) - COUNT(result_value) AS 결측행수
FROM lab
""")

## 연습문제

In [ ]:
# [연습 Q1] 처방의 평균 투약일수는?
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q2] 맥박을 서맥(<60)·정상·빈맥(>100)으로 분류해 보라
# 힌트: CASE
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q3] HbA1c 검사의 최솟값·최댓값·평균은?
# 힌트: WHERE test_name='HbA1c'
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q4] HbA1c가 7.0 이상인 검사 건수의 비율은?
# 힌트: CASE×SUM, 100.0
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q5] 입원 재원일수가 14일을 넘는 내원은 몇 건인가?
q("""
-- 여기에 SQL을 작성하세요

""")

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**A1. 평균 투약일수**
```sql
SELECT ROUND(AVG(duration_days),1) FROM prescription;
```

**A2. 맥박 분류**
```sql
SELECT visit_id, pulse, CASE WHEN pulse < 60 THEN '서맥' WHEN pulse > 100 THEN '빈맥' ELSE '정상' END AS grade FROM vital;
```

**A3. HbA1c 요약**
```sql
SELECT MIN(result_value), MAX(result_value), ROUND(AVG(result_value),1) FROM lab WHERE test_name='HbA1c';
```

**A4. HbA1c≥7.0 비율**
```sql
SELECT ROUND(100.0*SUM(CASE WHEN result_value>=7.0 THEN 1 ELSE 0 END)/COUNT(*),1) FROM lab WHERE test_name='HbA1c';
```

**A5. 14일 초과 입원**
```sql
SELECT COUNT(*) FROM visit WHERE visit_type='입원' AND julianday(discharge_date)-julianday(visit_date) > 14;
```